Tema 2 - Perceptron

In [ ]:
import numpy as np
import math

train = np.load("./nn-2026-fii-homework-1/train.npz")
test = np.load("./nn-2026-fii-homework-1/test.npz")

X_train = train["data"]
y_train = train["labels"]
X_test = test["data"]

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)

print(y_train)

X_train = X_train / 255.0
X_test = X_test / 255.0


W = np.random.randn(784, 10) * 0.01
b = np.zeros(10)

(60000, 784)
(60000,)
(15000, 784)
[9 3 6 ... 9 5 7]
[[0.2745098  0.         0.         ... 0.99215686 0.         0.21568627]
 [0.         0.         0.         ... 0.93333333 0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.42352941 0.         0.        ]
 [0.         0.         0.         ... 0.80784314 0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


Calculam Z-ul (care va fi o matrice de (m,10), fiind scorul pentru fiecare 10 perceptroni la toate imaginile), si apoi il trecem prin softmax pentru a afla Y_prob (care va fi o matrice tot de (m,10), de data asta fiind probabilitatile pentru fiecare valoare 0-9, pentru fiecare imagine)

In [47]:
def softmax(Z:np.array) -> np.array:
    Y = np.zeros((Z.shape[0],10))
    line = 0
    column = 0
    for row in Z:
        ez_list = []
        for z in row:
            ez_list.append(math.exp(z))
        sum_ez = sum(ez_list)
        for z in row:
            yi = ez_list[column]/sum_ez
            Y[line][column] = yi
            column += 1
        line += 1
        column = 0
    return Y

# Y_hat va fi matrice de (m,1) si va stabili rezultatatele prezise la fiecare imagine
def predicted_class_matrix(Y:np.array) -> np.array:
    Y_hat = np.zeros((Y.shape[0],1))
    Y_hat = np.argmax(Y,axis=1,keepdims=True)
    return Y_hat
# Z will be of size (m,10)
def forward_propagation(X:np.array, W:np.array, b:np.array) -> np.array:
    Z = np.dot(X,W) + b
    # print(Z.shape)
    # print(Z)
    Y = softmax(Z)
    return Y

Y_prob = forward_propagation(X_train, W, b)
print(Y_prob)

Y_predicted = predicted_class_matrix(Y_prob)
print()
print(Y_predicted)


[[0.09350754 0.10883982 0.10090869 ... 0.09395083 0.09370475 0.10774337]
 [0.10384041 0.10240805 0.09740187 ... 0.09161288 0.09440343 0.11633284]
 [0.10783411 0.10943531 0.08886777 ... 0.09342534 0.09760867 0.09652264]
 ...
 [0.10330878 0.10573455 0.09767455 ... 0.09162268 0.08477593 0.10116202]
 [0.09514171 0.10053834 0.1071998  ... 0.08951892 0.09707333 0.0991778 ]
 [0.09320066 0.10687707 0.10386297 ... 0.0985919  0.08961406 0.09762692]]

[[3]
 [3]
 [3]
 ...
 [4]
 [6]
 [3]]


Calculam CrossEntropy pentru a vedea eroarea. cross_entropy va returna eroarea calculata pentru toti cei 10 perceptroni (care este suma din produs vector target (care ar fi [0,0,1,0,...,0]) cu log(probabilitatea prezicerii de pe acel perceptron))

CrossEntropy este, pentru o imagine, suma din elementul din target(adica daca e cel prezis sau nu)*log(probabilitatea prezisa la acel index).

Dar pentru toate elementele mai putin target-ul meu corect, elementele din suma sunt zero, deci cross entropy ar fi -log(probabilitate_prezicere_corecta)

iar crossentropy pe tot setul vine -log(probabilitate_prezicere_corecta)/m

In [48]:
#putem sa ne folosim direct de y_train care e de forma [9,3,2,...., etc.] pentru a extrage indexul probabilitatii prezicerii corecte rapid
def cross_entropy(Y_prob:np.array, y_train:np.array)->np.array:
    m = y_train.shape[0]
    ## extrag valorile din y_prob de la 0 la m-1 la indicii din y_train
    probabilitati_corecte = Y_prob[np.arange(m), y_train]
    loss = -np.sum(np.log(probabilitati_corecte)) / m
    return loss

loss = cross_entropy(Y_prob, y_train)
print(loss)
    

2.305048041118812


Trecem la gradient descend. Vom calcula target, care este o matrice, fiecare linie avand forma [0,1,0,0,...,0], daca predictia mea ar trebui sa dea 2

Target-y va fi aceasta matrice minus matricea probabilitatilor, si da tot o matrice (m,10)

In [49]:
def get_target(y_train:np.array) -> np.array:
    Target = np.zeros((y_train.shape[0],10))
    line=0
    column=0
    for label in y_train:
        for column in range(0,10):
            if column == label:
                Target[line][column] = 1
            else:
                Target[line][column] = 0
        line += 1
    return Target

def gradient_descend(Target:np.array, Y_prob:np.array, learning_rate:float, X_train:np.array, W:np.array, b:np.array):
    gradient = Target - Y_prob
    transpose = np.transpose(X_train)

    m = X_train.shape[0]
    change_w = (learning_rate * np.dot(transpose,gradient))/m
    W = W + change_w
    change_b = (learning_rate * np.sum(gradient, axis=0))/m
    b = b + change_b
    return (W,b)

print(W)
print()
print(b)

Target = get_target(y_train)
(W,b) = gradient_descend(Target,Y_prob,0.01,X_train,W,b)

print(W)
print()

print(b)


[[ 0.00129955  0.00841719  0.00717844 ...  0.01691101 -0.00148804
  -0.00629927]
 [ 0.0063127   0.00721574 -0.02153396 ...  0.00281589 -0.00493446
   0.0152498 ]
 [-0.00091002 -0.01158589  0.01287012 ... -0.00373801 -0.00320882
  -0.01375315]
 ...
 [-0.00094492 -0.00090013 -0.00055343 ... -0.0153939  -0.0066313
   0.00010452]
 [-0.00668012 -0.00026536  0.00020389 ... -0.00683015 -0.00743447
   0.01392274]
 [-0.00766772  0.00862505  0.0070643  ... -0.00136251 -0.01091197
   0.00396945]]

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[[ 1.04755259e-03  8.24299561e-03  7.82742924e-03 ...  1.70436212e-02
  -1.01738988e-03 -6.30697565e-03]
 [ 6.31357829e-03  7.21626898e-03 -2.15341251e-02 ...  2.81573609e-03
  -4.93462474e-03  1.52496174e-02]
 [-9.76077978e-04 -1.16570860e-02  1.28021174e-02 ... -3.76177839e-03
  -3.18629393e-03 -1.37630429e-02]
 ...
 [-9.20604793e-04 -9.18452292e-04 -8.28378273e-04 ... -1.53989644e-02
  -6.61379119e-03 -2.93831659e-06]
 [-6.73660558e-03 -3.09504916e-04  1.46330892e-04 .

Acum trecem la loop de antrenare

In [54]:
W = np.random.randn(784, 10) * 0.01
b = np.zeros(10)

epoches = 50
learning_rate = 0.5

Target = get_target(y_train)

print("Starting training...")

for epoch in range(0,epoches+1):
    # forward propagation

    Y_prob = forward_propagation(X_train,W,b)

    # backward propagation

    W,b = gradient_descend(Target,Y_prob,learning_rate,X_train, W, b)

    #printuri generale

    loss = cross_entropy(Y_prob, y_train)
    # calculez acuratetea
    
    Y_hat = predicted_class_matrix(Y_prob)
    # Y_hat are forma (m, 1), iar y_train are forma (m,). 
    # flatten() pentru a le aduce la aceeasi forma inainte de comparare
    accuracy = np.mean(Y_hat.flatten() == y_train)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Accuracy: {accuracy * 100:.2f}%")

print("Finished training")

Starting training...
Epoch 000 | Loss: 2.3197 | Accuracy: 6.51%
Epoch 010 | Loss: 0.7824 | Accuracy: 83.98%
Epoch 020 | Loss: 0.6026 | Accuracy: 86.11%
Epoch 030 | Loss: 0.5291 | Accuracy: 87.06%
Epoch 040 | Loss: 0.4872 | Accuracy: 87.70%
Epoch 050 | Loss: 0.4593 | Accuracy: 88.21%
Finished training


Testam datele

In [55]:
import pandas as pd

Y_prob_test = forward_propagation(X_test, W, b)

Y_predicted_test = predicted_class_matrix(Y_prob_test)
submission = pd.DataFrame({
    "ID": range(len(Y_predicted_test)),
    "target": Y_predicted_test.flatten()
})

submission.to_csv("submission.csv", index=False)
print("Submission file created")


Submission file created
